%md
### Customer Segmentation
Step 1: Calculate Raw RFM Values

RFM is the industry-standard method for customer segmentation. 
- R = Recency (how recently did they buy?)
- F = Frequency (how often do they buy?) ``
- M = Monetary (how much do they spend?). 


In [0]:
%sql
CREATE OR REPLACE TABLE gold.customer_rfm_base
USING DELTA
AS
SELECT
  c.customer_key,
  c.full_name,
  c.country,
  c.gender,
  c.marital_status,
  c.age_years,
 
  -- Recency: days since their last purchase
  -- Smaller number = more recent = better customer
  DATEDIFF(MAX(s.order_date), CURRENT_DATE()) * -1  AS recency_days,
 
  -- Frequency: how many separate orders they placed
  COUNT(DISTINCT s.order_number)                    AS frequency,
 
  -- Monetary: total amount they have ever spent
  ROUND(SUM(s.sales_amount), 2)                     AS monetary_value,
  ROUND(AVG(s.sales_amount), 2)                     AS avg_order_value,
 
  MIN(s.order_date)                                 AS first_purchase_date,
  MAX(s.order_date)                                 AS last_purchase_date
 
FROM workspace.silver.sales s
JOIN workspace.silver.customers c
  ON s.customer_key = c.customer_key
 
GROUP BY
  c.customer_key, c.full_name, c.country,
  c.gender, c.marital_status, c.age_years;




Step 2: Score Each Customer 1 to 4_

In [0]:
%sql
CREATE OR REPLACE TABLE gold.customer_rfm_scored
USING DELTA
AS
SELECT
  *,
 
  -- Recency score: bought most recently = score 4
  NTILE(4) OVER (ORDER BY recency_days DESC)        AS r_score,
 
  -- Frequency score: bought most often = score 4
  NTILE(4) OVER (ORDER BY frequency ASC)            AS f_score,
 
  -- Monetary score: spent the most = score 4
  NTILE(4) OVER (ORDER BY monetary_value ASC)       AS m_score
 
FROM gold.customer_rfm_base;

In [0]:
%sql
SELECT * FROM workspace.gold.customer_rfm_scored

Step 3 Assign Customer Segments

In [0]:
%sql
CREATE OR REPLACE TABLE gold.customer_segments
USING DELTA
AS
SELECT
  *,
  (r_score + f_score + m_score)                     AS total_rfm_score,
 
  CASE
    WHEN r_score = 4 AND f_score = 4 AND m_score = 4
      THEN 'Champion'         -- Best customers. Recent, frequent, big spenders
    WHEN r_score >= 3 AND f_score >= 3
      THEN 'Loyal Customer'   -- Buy regularly and recently
    WHEN r_score = 4 AND f_score <= 2
      THEN 'New Customer'     -- Bought recently but only once or twice
    WHEN r_score >= 3 AND m_score >= 3
      THEN 'High Spender'     -- Spends big but may not buy often
    WHEN r_score = 1 AND f_score >= 3
      THEN 'At Risk'          -- Used to buy often but gone quiet
    WHEN r_score = 1 AND f_score = 1
      THEN 'Lost Customer'    -- Not seen in a long time, low frequency
    ELSE 'Needs Attention'
  END                                               AS customer_segment
 
FROM workspace.gold.customer_rfm_scored
ORDER BY total_rfm_score DESC;



Step 4: Segment Summary for the Executive Dashboard

In [0]:

%sql

CREATE OR REPLACE TABLE gold.customer_segment_summary
USING DELTA
AS
SELECT
  customer_segment,
  COUNT(*)                                          AS total_customers,
  ROUND(AVG(monetary_value), 2)                     AS avg_spend,
  ROUND(SUM(monetary_value), 2)                     AS total_revenue_contributed,
  ROUND(AVG(frequency), 1)                          AS avg_orders,
  ROUND(AVG(recency_days), 0)                       AS avg_days_since_purchase
 
FROM workspace.gold.customer_segments
GROUP BY customer_segment
ORDER BY total_revenue_contributed DESC;
